In [ ]:
import geopandas as gpd
import earthaccess
from shapely.geometry import box
import datetime

# Import core modules of the firerx_ml framework (assuming j-gams/firerx_ml is cloned and env variables are configured)
from firerx_ml import utils
from firerx_ml import manage_data
from firerx_ml import models

In [ ]:

# ==========================================
# Phase 1: Multi-source STAC Data Ingestion (HLS + ECOSTRESS)
# ==========================================

# 1. Define Midwest Region of Interest (ROI) and time window
bbox_coords = (-87.5, 39.5, -86.5, 40.5) # [min_lon, min_lat, max_lon, max_lat]
roi_polygon = box(*bbox_coords)
time_window = ("2023-10-01", "2024-06-30")

# 2. Authentication
auth = earthaccess.login(persist=True)

# 3. Retrieve HLS structural data (HLSL30, HLSF30)
hls_results = earthaccess.search_data(
    short_name=["HLSL30", "HLSF30"],
    bounding_box=bbox_coords,
    temporal=time_window
)

# 4. Retrieve ECOSTRESS physiological data 
# ECO3ETALEXI: Daily Evapotranspiration (ET) based on the ALEXI model
# ECO2LSTE: Land Surface Temperature and Emissivity (LST)
eco_results = earthaccess.search_data(
    short_name=["ECO3ETALEXI", "ECO2LSTE"],
    bounding_box=bbox_coords,
    temporal=time_window
)

print(f"Retrieved {len(hls_results)} HLS granules and {len(eco_results)} ECOSTRESS granules.")



In [ ]:

# ==========================================
# Phase 2: Configure Alignment Rules and Build Data Pyramid
# ==========================================

# 1. Generate firerx_ml configuration file
# Align 30m HLS and 70m ECOSTRESS to a unified spatiotemporal grid
pyramid_config = utils.create_config(
    project_name="Midwest_CoverCrop_Fusion",
    spatial_resolution=30,           # Base resolution set to 30m (HLS-dominated)
    temporal_aggregation="14D",      # Bi-weekly composite (smooths irregular ECOSTRESS overpasses)
    crs="EPSG:32616",                # UTM Zone 16N (suitable for Indiana/Illinois)
    input_sources={
        "optical": {"data": hls_results, "bands": ["Red", "NIR", "Fmask"]},
        "thermal": {"data": eco_results, "bands": ["ETdaily", "LST", "QC"]}
    },
    target_labels="path_to_ground_truth_polygons.geojson" # Vector data containing WCC, Wheat, Fallow labels
)

# 2. Run the preprocessing pipeline to generate an "analysis-ready" data pyramid
# This step handles resampling, cloud masking/denoising, and raster alignment
pyramid_dataset_path = manage_data.build_pyramid(
    config=pyramid_config,
    output_dir="./data/pyramids/",
    compute_engine="dask"            # Use Dask for distributed parallel processing
)

print(f"Data Pyramid built successfully at: {pyramid_dataset_path}")



In [ ]:

# ==========================================
# Phase 3: Deep Learning Model Engine Training
# ==========================================

# 1. Define Pyramid-ready Vision Transformer (ViT) model architecture
# Combine temporal structural features (NDVI) with physiological features (ET/LST)
model_config = utils.create_model_config(
    architecture="ViT_Pyramid",
    input_channels=4,                # Example: Red, NIR, ET, LST
    sequence_length=19,              # 14-day step, approx. 19 time steps over 9 months
    num_classes=3,                   # Classes: [WCC, Winter_Wheat, Fallow/Weeds]
    batch_size=64,
    learning_rate=1e-4,
    epochs=50
)

# 2. Initialize model
fusion_model = models.initialize_model(model_config)

# 3. Load dataset and split into training/validation sets
train_loader, val_loader = manage_data.get_dataloaders(
    pyramid_path=pyramid_dataset_path,
    split_ratio=(0.8, 0.2)
)

# 4. Execute training and evaluation
training_results = models.train_and_evaluate(
    model=fusion_model,
    train_data=train_loader,
    val_data=val_loader,
    save_checkpoint_dir="./models/checkpoints/"
)

# 5. Output performance metrics (Confusion Matrix, Precision, Recall)
models.plot_metrics(training_results)
print("Fusion model training complete. Best Validation Accuracy:", training_results['best_val_acc'])